In [2]:
# fred_cds_spread_presentation.py

import os
from fredapi import Fred
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, linregress
from pptx import Presentation
from pptx.util import Inches

# ==============================
# 1. CONFIGURATION
# ==============================

FRED_API_KEY = os.getenv("FRED_API_KEY")

series = {
    "ice_oas": "BAMLC0A0CM",   # ICE BofA US Corporate Index OAS
    "baa_spread": "BAA10Y"     # Moody's Baa minus 10Y Treasury
}

start_date = "2014-01-01"
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

rolling_window = 90
pptx_out = "CDS_vs_Corporate_Spreads_FRED_API.pptx"

# ==============================
# 2. DOWNLOAD DATA VIA FRED API
# ==============================

fred = Fred(api_key=FRED_API_KEY)

data = {}
for name, sid in series.items():
    data[name] = fred.get_series(
        sid,
        observation_start=start_date,
        observation_end=end_date
    )

df = pd.DataFrame(data)
df = df.dropna()

# Convert percent to basis points
df_bps = df * 100

# ==============================
# 3. ANALYSIS
# ==============================

x = df_bps["ice_oas"]
y = df_bps["baa_spread"]

pearson_r, pval = pearsonr(x, y)
slope, intercept, r_val, p_reg, std_err = linregress(x, y)

rolling_corr = x.rolling(rolling_window).corr(y)

# ==============================
# 4. PLOTS
# ==============================

# Time series
plt.figure(figsize=(10,5))
plt.plot(df_bps.index, x, label="ICE OAS (bps)")
plt.plot(df_bps.index, y, label="Baa Spread (bps)")
plt.legend()
plt.title("Corporate Bond Spreads (bps)")
plt.tight_layout()
plt.savefig("timeseries.png")
plt.close()

# Scatter
plt.figure(figsize=(6,6))
plt.scatter(x, y, alpha=0.3, s=10)
xs = np.linspace(x.min(), x.max(), 100)
plt.plot(xs, intercept + slope*xs)
plt.title("Scatter: OAS vs Baa Spread")
plt.xlabel("ICE OAS (bps)")
plt.ylabel("Baa Spread (bps)")
plt.tight_layout()
plt.savefig("scatter.png")
plt.close()

# Rolling correlation
plt.figure(figsize=(10,4))
plt.plot(rolling_corr.index, rolling_corr)
plt.axhline(0, linestyle="--")
plt.title(f"{rolling_window}-Day Rolling Correlation")
plt.tight_layout()
plt.savefig("rolling_corr.png")
plt.close()

# ==============================
# 5. BUILD POWERPOINT
# ==============================

prs = Presentation()

# Title Slide
slide = prs.slides.add_slide(prs.slide_layouts[0])
slide.shapes.title.text = "Credit Default Swap Proxy vs Corporate Bond Spreads"
subtitle = slide.placeholders[1]
subtitle.text = f"FRED API Data | {start_date} to {end_date}"

# Data Slide
slide = prs.slides.add_slide(prs.slide_layouts[1])
slide.shapes.title.text = "Data & Methodology"
tf = slide.shapes.placeholders[1].text_frame
tf.text = (
    "Data Source: FRED API\n"
    "- BAMLC0A0CM (ICE BofA OAS)\n"
    "- BAA10Y (Moody's Baa spread)\n\n"
    "Methods:\n"
    "- Pearson correlation\n"
    "- OLS regression\n"
    "- Rolling correlation\n"
)

# Time Series Slide
slide = prs.slides.add_slide(prs.slide_layouts[5])
slide.shapes.title.text = "Time Series (bps)"
slide.shapes.add_picture("timeseries.png", Inches(0.5), Inches(1.3), height=Inches(4.5))

# Scatter Slide
slide = prs.slides.add_slide(prs.slide_layouts[5])
slide.shapes.title.text = "Cross-Sectional Relationship"
slide.shapes.add_picture("scatter.png", Inches(0.5), Inches(1.3), height=Inches(4.5))

# Rolling Correlation Slide
slide = prs.slides.add_slide(prs.slide_layouts[5])
slide.shapes.title.text = "Rolling Correlation"
slide.shapes.add_picture("rolling_corr.png", Inches(0.5), Inches(1.3), height=Inches(4.5))

# Results Slide
slide = prs.slides.add_slide(prs.slide_layouts[1])
slide.shapes.title.text = "Statistical Results"
tf = slide.shapes.placeholders[1].text_frame
tf.text = (
    f"Pearson Correlation: {pearson_r:.3f}\n"
    f"P-value: {pval:.2e}\n"
    f"OLS Slope: {slope:.3f}\n"
    f"Observations: {len(df_bps)}"
)

prs.save(pptx_out)

print("Presentation created:", pptx_out)

Presentation created: CDS_vs_Corporate_Spreads_FRED_API.pptx
